# Data preparation
## 1. Environment and paths

In [1]:
import os
import subprocess
import sys

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

# Project root directory
PROJECT_ROOT = r"C:\Users\b1795\Desktop\Laboratory2_SP"

DATA_RAW_DIR = os.path.join(PROJECT_ROOT, "data_raw")
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data_processed")
CV_DIR = os.path.join(DATA_PROCESSED_DIR, "cv_folds")

os.makedirs(DATA_RAW_DIR, exist_ok=True)
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
os.makedirs(CV_DIR, exist_ok=True)

# Part A outputs
POS_FASTA = os.path.join(DATA_RAW_DIR, "positive.fasta")
NEG_FASTA = os.path.join(DATA_RAW_DIR, "negative.fasta")
POS_TSV = os.path.join(DATA_RAW_DIR, "positive.tsv")
NEG_TSV = os.path.join(DATA_RAW_DIR, "negative.tsv")

# Part B outputs
# MMseqs output directory
MMSEQS_POS_DIR = os.path.join(DATA_RAW_DIR, "mmseqs_positive")
MMSEQS_NEG_DIR = os.path.join(DATA_RAW_DIR, "mmseqs_negative")
os.makedirs(MMSEQS_POS_DIR, exist_ok=True)
os.makedirs(MMSEQS_NEG_DIR, exist_ok=True)

# non-redundant TSV
POS_NR_TSV = os.path.join(DATA_PROCESSED_DIR, "positive_nr.tsv")
NEG_NR_TSV = os.path.join(DATA_PROCESSED_DIR, "negative_nr.tsv")

# Training / benchmark sets
TRAIN_POS_TSV = os.path.join(DATA_PROCESSED_DIR, "training_positive.tsv")
TRAIN_NEG_TSV = os.path.join(DATA_PROCESSED_DIR, "training_negative.tsv")
BENCH_POS_TSV = os.path.join(DATA_PROCESSED_DIR, "benchmark_positive.tsv")
BENCH_NEG_TSV = os.path.join(DATA_PROCESSED_DIR, "benchmark_negative.tsv")

# Training set with fold information
TRAIN_WITH_FOLD = os.path.join(DATA_PROCESSED_DIR, "training_with_fold_info.tsv")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW_DIR:", DATA_RAW_DIR)
print("DATA_PROCESSED_DIR:", DATA_PROCESSED_DIR)

# Check whether mmseqs2 is available in PATH
def check_mmseqs_available():
    try:
        result = subprocess.run(
            ["mmseqs"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=True,
        )
        print("mmseqs2 is available in PATH.")
        return True
    except Exception as e:
        print("WARNING: mmseqs2 not found in PATH.")
        print("Please install mmseqs2 and make sure 'mmseqs' command is available.")
        print("See: https://github.com/soedinglab/MMseqs2")
        return False

MMSEQS_AVAILABLE = check_mmseqs_available()

PROJECT_ROOT: C:\Users\b1795\Desktop\Laboratory2_SP
DATA_RAW_DIR: C:\Users\b1795\Desktop\Laboratory2_SP\data_raw
DATA_PROCESSED_DIR: C:\Users\b1795\Desktop\Laboratory2_SP\data_processed
mmseqs2 is available in PATH.


## 2. Run MMseqs2 clustering (positive and negative separately)  

In [2]:
def run_mmseqs_easy_cluster(input_fasta: str, out_dir: str, prefix: str):
    if not MMSEQS_AVAILABLE:
        raise RuntimeError("mmseqs2 is not available. Install it before running clustering.")

    os.makedirs(out_dir, exist_ok=True)
    rep_fasta = os.path.join(out_dir, f"{prefix}_rep_seq.fasta")
    cluster_tsv = os.path.join(out_dir, f"{prefix}_cluster.tsv")

    if os.path.exists(rep_fasta) and os.path.exists(cluster_tsv):
        print(f"[mmseqs] {prefix} clustering outputs already exist. Skipping mmseqs run.")
        return rep_fasta, cluster_tsv

    tmp_dir = os.path.join(out_dir, "tmp")
    os.makedirs(tmp_dir, exist_ok=True)

    cmd = [
        "mmseqs",
        "easy-cluster",
        input_fasta,
        os.path.join(out_dir, prefix),
        tmp_dir,
        "--min-seq-id", "0.3",
        "-c", "0.4",
        "--cov-mode", "0",
        "--cluster-mode", "1",
    ]

    print("Running mmseqs:", " ".join(cmd))
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    if result.returncode != 0:
        print("mmseqs stderr:\n", result.stderr)
        raise RuntimeError("mmseqs easy-cluster failed.")

    print("mmseqs stdout:\n", result.stdout)
    print(f"[mmseqs] Finished clustering for {input_fasta}")
    print(f"[mmseqs] Representative FASTA: {rep_fasta}")
    print(f"[mmseqs] Cluster TSV: {cluster_tsv}")
    return rep_fasta, cluster_tsv

# Run clustering for positive / negative datasets
if MMSEQS_AVAILABLE:
    POS_REP_FASTA, POS_CLUSTER_TSV = run_mmseqs_easy_cluster(
        POS_FASTA, MMSEQS_POS_DIR, "positive_clusters"
    )
    NEG_REP_FASTA, NEG_CLUSTER_TSV = run_mmseqs_easy_cluster(
        NEG_FASTA, MMSEQS_NEG_DIR, "negative_clusters"
    )
else:
    POS_REP_FASTA = NEG_REP_FASTA = None
    POS_CLUSTER_TSV = NEG_CLUSTER_TSV = None

[mmseqs] positive_clusters clustering outputs already exist. Skipping mmseqs run.
[mmseqs] negative_clusters clustering outputs already exist. Skipping mmseqs run.


The MMseqs2 easy-cluster function invokes internal .sh scripts on Windows, which cannot be executed directly, leading to errors when run locally.  
To ensure compatibility, this project installs and runs the Linux version of MMseqs2 inside WSL (Ubuntu).  
The positive and negative clustering result files (rep_seq.fasta and cluster.tsv) were generated externally in advance.  
Therefore, if these output files are detected, this code block automatically skips MMseqs2 execution and proceeds to downstream data processing.

## 3. Build non-redundant TSVs from MMseqs representatives  

In [3]:
def load_representatives(cluster_tsv: str) -> set:
    df = pd.read_csv(cluster_tsv, sep="\t", header=None, names=["member", "representative"])
    rep_ids = set(df["representative"].unique())
    print(f"{cluster_tsv}: {len(rep_ids)} representatives")
    return rep_ids

def filter_tsv_by_reps(input_tsv: str, rep_ids: set, output_tsv: str):
    df = pd.read_csv(input_tsv, sep="\t")
    if "accession" not in df.columns:
        raise ValueError(f"'accession' column not found in {input_tsv}")
    before = len(df)
    df_nr = df[df["accession"].isin(rep_ids)].copy()
    after = len(df_nr)
    df_nr.to_csv(output_tsv, sep="\t", index=False)
    print(f"{input_tsv}: {before} rows -> {after} non-redundant rows")
    print("Saved:", output_tsv)
    return df_nr

if MMSEQS_AVAILABLE:
    # positive
    pos_reps = load_representatives(POS_CLUSTER_TSV)
    df_pos_nr = filter_tsv_by_reps(POS_TSV, pos_reps, POS_NR_TSV)
    # negative
    neg_reps = load_representatives(NEG_CLUSTER_TSV)
    df_neg_nr = filter_tsv_by_reps(NEG_TSV, neg_reps, NEG_NR_TSV)
else:
    print("Skip building non-redundant TSVs because mmseqs2 is not available.")

C:\Users\b1795\Desktop\Laboratory2_SP\data_raw\mmseqs_positive\positive_clusters_cluster.tsv: 2935 representatives
C:\Users\b1795\Desktop\Laboratory2_SP\data_raw\positive.tsv: 2935 rows -> 2935 non-redundant rows
Saved: C:\Users\b1795\Desktop\Laboratory2_SP\data_processed\positive_nr.tsv
C:\Users\b1795\Desktop\Laboratory2_SP\data_raw\mmseqs_negative\negative_clusters_cluster.tsv: 20615 representatives
C:\Users\b1795\Desktop\Laboratory2_SP\data_raw\negative.tsv: 20615 rows -> 20615 non-redundant rows
Saved: C:\Users\b1795\Desktop\Laboratory2_SP\data_processed\negative_nr.tsv


## 4. Split non-redundant datasets into training (80%) and benchmarking (20%)  

In [4]:
RANDOM_SEED = 42
rng = np.random.RandomState(RANDOM_SEED)

def split_train_benchmark(df: pd.DataFrame, train_frac: float = 0.8):
    idx = np.arange(len(df))
    rng.shuffle(idx)
    split_point = int(len(idx) * train_frac)
    train_idx = idx[:split_point]
    bench_idx = idx[split_point:]
    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_bench = df.iloc[bench_idx].reset_index(drop=True)
    return df_train, df_bench

# Load non-redundant TSV files (if already loaded as df_pos_nr / df_neg_nr in the previous section, they can be reused)
df_pos_nr = pd.read_csv(POS_NR_TSV, sep="\t")
df_neg_nr = pd.read_csv(NEG_NR_TSV, sep="\t")

# Positive samples
df_pos_train, df_pos_bench = split_train_benchmark(df_pos_nr, train_frac=0.8)
df_pos_train.to_csv(TRAIN_POS_TSV, sep="\t", index=False)
df_pos_bench.to_csv(BENCH_POS_TSV, sep="\t", index=False)
print(f"Positive: total={len(df_pos_nr)}, train={len(df_pos_train)}, bench={len(df_pos_bench)}")

# Negative samples
df_neg_train, df_neg_bench = split_train_benchmark(df_neg_nr, train_frac=0.8)
df_neg_train.to_csv(TRAIN_NEG_TSV, sep="\t", index=False)
df_neg_bench.to_csv(BENCH_NEG_TSV, sep="\t", index=False)
print(f"Negative: total={len(df_neg_nr)}, train={len(df_neg_train)}, bench={len(df_neg_bench)}")
print("Training / benchmarking TSVs saved to:", DATA_PROCESSED_DIR)

Positive: total=2935, train=2348, bench=587
Negative: total=20615, train=16492, bench=4123
Training / benchmarking TSVs saved to: C:\Users\b1795\Desktop\Laboratory2_SP\data_processed


## 5. Build 5-fold cross-validation subsets (stratified)

In [5]:
# Reload training sets (or reuse df_pos_train / df_neg_train from the previous section)
df_pos_train = pd.read_csv(TRAIN_POS_TSV, sep="\t")
df_neg_train = pd.read_csv(TRAIN_NEG_TSV, sep="\t")

# Add label column to positive and negative samples
df_pos_train = df_pos_train.copy()
df_neg_train = df_neg_train.copy()
df_pos_train["label"] = 1  # positive
df_neg_train["label"] = 0  # negative

# Merge into a single combined training table
df_train_all = pd.concat([df_pos_train, df_neg_train], ignore_index=True)
print("Combined training size:", len(df_train_all))
print("Positive:", (df_train_all["label"] == 1).sum(),
        "Negative:", (df_train_all["label"] == 0).sum())

# Perform 5-fold splitting using StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
fold_ids = np.zeros(len(df_train_all), dtype=int)

X_dummy = np.zeros((len(df_train_all), 1))
y = df_train_all["label"].values

for fold_idx, (_, test_index) in enumerate(skf.split(X_dummy, y), start=1):
    fold_ids[test_index] = fold_idx

df_train_all["fold"] = fold_ids

# Write each fold to a separate TSV file
for fold_idx in range(1, 6):
    df_fold = df_train_all[df_train_all["fold"] == fold_idx].copy()
    out_path = os.path.join(CV_DIR, f"cv{fold_idx}.tsv")
    df_fold.to_csv(out_path, sep="\t", index=False)
    print(f"Fold {fold_idx}: {len(df_fold)} samples -> {out_path}")

# Save the complete training table with fold information
df_train_all.to_csv(TRAIN_WITH_FOLD, sep="\t", index=False)
print("Training with fold info saved to:", TRAIN_WITH_FOLD)

Combined training size: 18840
Positive: 2348 Negative: 16492
Fold 1: 3768 samples -> C:\Users\b1795\Desktop\Laboratory2_SP\data_processed\cv_folds\cv1.tsv
Fold 2: 3768 samples -> C:\Users\b1795\Desktop\Laboratory2_SP\data_processed\cv_folds\cv2.tsv
Fold 3: 3768 samples -> C:\Users\b1795\Desktop\Laboratory2_SP\data_processed\cv_folds\cv3.tsv
Fold 4: 3768 samples -> C:\Users\b1795\Desktop\Laboratory2_SP\data_processed\cv_folds\cv4.tsv
Fold 5: 3768 samples -> C:\Users\b1795\Desktop\Laboratory2_SP\data_processed\cv_folds\cv5.tsv
Training with fold info saved to: C:\Users\b1795\Desktop\Laboratory2_SP\data_processed\training_with_fold_info.tsv


## 6. Quick sanity check  

In [6]:
df_train_all = pd.read_csv(TRAIN_WITH_FOLD, sep="\t")
print(df_train_all.groupby(["fold", "label"]).size().unstack(fill_value=0))

label     0    1
fold            
1      3298  470
2      3298  470
3      3298  470
4      3299  469
5      3299  469
